# Tarea 01 · Comprender los datos y formular un problema de predicción

**Pregunta de trabajo:** ¿un remate terminará en gol?

Este notebook usa eventos reales de un partido de fútbol para practicar exploración de datos, clasificación de atributos, tareas de minería y evaluación de una regla sencilla. No se entrena un modelo predictivo. Ejecuta todas las celdas en orden.

## Instalación de dependencias

En una terminal, desde la carpeta del repositorio, instala las bibliotecas utilizadas:

```bash
python -m pip install pandas jupyter ipykernel
```

En Google Colab, pandas ya está disponible. Si hace falta, ejecuta `!pip install pandas` en una celda. El notebook descarga los datos por internet durante su ejecución y guarda una copia local reutilizable.

**Fuente: StatsBomb Open Data.** Los datos y resultados de este análisis son para fines educativos. StatsBomb solicita indicar la fuente y mostrar su logotipo al publicar análisis basados en estos datos. Las conclusiones son nuestras y no representan opiniones de StatsBomb.

![Logotipo de StatsBomb](https://raw.githubusercontent.com/hudl/open-data/4b73468fc5b0f1950f9f66fada70ad3a4f9327cb/img/SB%20-%20Icon%20Lockup%20-%20Colour%20positive.png)


## 1. Descarga y lectura de los datos

Primero fijamos la versión pública de StatsBomb Open Data y definimos rutas de descarga. La celda siguiente importa las bibliotecas básicas, fija la selección y crea una carpeta de caché. Si los archivos ya existen, se reutilizan; de lo contrario, el notebook los descarga desde GitHub.

In [1]:
from pathlib import Path
from urllib.request import urlopen
import json
import pandas as pd
from IPython.display import display

# Usamos una versión fija para que los archivos no cambien entre ejecuciones.
version_datos = "4b73468fc5b0f1950f9f66fada70ad3a4f9327cb"
competicion_id = 43
temporada_id = 106
nombre_notebook = "tarea_01_datos_y_prediccion"

# Buscamos la raíz del repositorio a partir de la carpeta desde donde se abrió Jupyter.
raiz = Path.cwd()
while not (raiz / "notebooks" / (nombre_notebook + ".ipynb")).exists():
    if raiz == raiz.parent:
        raise FileNotFoundError("Abre este notebook desde una copia del repositorio.")
    raiz = raiz.parent

carpeta_datos = raiz / "data" / "raw" / nombre_notebook
carpeta_datos.mkdir(parents=True, exist_ok=True)
print("Carpeta de datos:", carpeta_datos)
print("Versión de StatsBomb Open Data:", version_datos)

Carpeta de datos: /home/luchosqi/Documentos/Universidad/semestres/Semestre 6/Ing Datos/repofutbol/IngDatos-PrediccionRemates/data/raw/tarea_01_datos_y_prediccion
Versión de StatsBomb Open Data: 4b73468fc5b0f1950f9f66fada70ad3a4f9327cb


Esta función pequeña descarga un archivo si no está en la caché local y devuelve su contenido como texto. Las rutas de StatsBomb siguen una estructura fija: competiciones, partidos y eventos. Fijamos la versión del repositorio en la URL para poder repetir la selección.

In [2]:
def leer_archivo_statsbomb(ruta):
    """Lee un archivo JSON de una versión fija de StatsBomb Open Data."""
    archivo_local = carpeta_datos / ruta
    if archivo_local.exists():
        texto = archivo_local.read_text(encoding="utf-8")
    else:
        url = "https://raw.githubusercontent.com/hudl/open-data/" + version_datos + "/" + ruta
        with urlopen(url, timeout=60) as respuesta:
            texto = respuesta.read().decode("utf-8")
        archivo_local.parent.mkdir(parents=True, exist_ok=True)
        archivo_local.write_text(texto, encoding="utf-8")
    return texto

# Leemos la tabla de competiciones y temporadas que publica StatsBomb.
texto_competiciones = leer_archivo_statsbomb("data/competitions.json")
registros_competiciones = json.loads(texto_competiciones)
competiciones = pd.json_normalize(registros_competiciones)
seleccion_competicion = competiciones[
    (competiciones["competition_id"] == competicion_id)
    & (competiciones["season_id"] == temporada_id)
]
display(seleccion_competicion[["competition_name", "season_name", "competition_id", "season_id"]])

,competition_name,season_name,competition_id,season_id
30,FIFA World Cup,2022,43,106


La tabla confirma la competición y temporada seleccionadas. El Mundial masculino de 2022 es una selección explícita, no una muestra aleatoria de todos los torneos disponibles.

Ahora leemos el archivo de partidos de esa competición y temporada. Seleccionamos la final Argentina–Francia por los nombres de los equipos, y mostramos identificador, fecha y participantes antes de pedir sus eventos.

In [3]:
ruta_partidos = "data/matches/" + str(competicion_id) + "/" + str(temporada_id) + ".json"
registros_partidos = json.loads(leer_archivo_statsbomb(ruta_partidos))
partidos = pd.json_normalize(registros_partidos)
final = partidos[
    ((partidos["home_team.home_team_name"] == "Argentina")
     & (partidos["away_team.away_team_name"] == "France"))
    | ((partidos["home_team.home_team_name"] == "France")
       & (partidos["away_team.away_team_name"] == "Argentina"))
]
partido = final.iloc[0]
partido_id = int(partido["match_id"])
identificacion = pd.DataFrame([{
    "partido_id": partido_id,
    "fecha": partido["match_date"],
    "competicion": "Mundial masculino 2022",
    "local": partido["home_team.home_team_name"],
    "visita": partido["away_team.away_team_name"]
}])
display(identificacion)

# Descargamos y leemos los eventos del partido seleccionado.
ruta_eventos = "data/events/" + str(partido_id) + ".json"
registros_eventos = json.loads(leer_archivo_statsbomb(ruta_eventos))
print("Cantidad de eventos registrados:", len(registros_eventos))

,partido_id,fecha,competicion,local,visita
0,3869685,2022-12-18,Mundial masculino 2022,Argentina,France


Cantidad de eventos registrados: 4407


La final seleccionada es Argentina–Francia, disputada el 18 de diciembre de 2022 (partido 3869685). El número de eventos del archivo aparece en la salida anterior. Un **evento** es una acción registrada en un instante del partido, como un pase, una recuperación o un remate. Por eso, contar eventos no equivale a contar jugadores ni partidos: una persona participa en muchos eventos y un partido contiene miles de ellos. Cada fila del archivo representa un evento, no una entidad única del tipo jugador o encuentro.

Mostramos un evento completo para observar su forma JSON. El ejemplo se toma directamente del archivo leído, sin reducirlo ni modificarlo.

In [4]:
evento_original = registros_eventos[0]
print(json.dumps(evento_original, ensure_ascii=False, indent=2))

{
  "id": "0584ee21-e3dd-4d9f-95a0-5b5e84be25c3",
  "index": 1,
  "period": 1,
  "timestamp": "00:00:00.000",
  "minute": 0,
  "second": 0,
  "type": {
    "id": 35,
    "name": "Starting XI"
  },
  "possession": 1,
  "possession_team": {
    "id": 779,
    "name": "Argentina"
  },
  "play_pattern": {
    "id": 1,
    "name": "Regular Play"
  },
  "team": {
    "id": 779,
    "name": "Argentina"
  },
  "duration": 0.0,
  "tactics": {
    "formation": 433,
    "lineup": [
      {
        "player": {
          "id": 6909,
          "name": "Damián Emiliano Martínez"
        },
        "position": {
          "id": 1,
          "name": "Goalkeeper"
        },
        "jersey_number": 23
      },
      {
        "player": {
          "id": 29201,
          "name": "Nahuel Molina Lucero"
        },
        "position": {
          "id": 2,
          "name": "Right Back"
        },
        "jersey_number": 26
      },
      {
        "player": {
          "id": 20572,
          "name": "Crist

En este evento hay campos **simples**, como `id`, `period` y `minute`, que guardan un valor directo. También hay campos **anidados**, como `type`, `team` y `player`, que contienen objetos con más campos. La ubicación es una lista de coordenadas. La siguiente celda accede a un campo simple y a uno anidado; los campos pueden no estar en todos los eventos, por eso usamos `get`.

In [5]:
# Ejemplos de acceso a un campo simple y a un campo anidado.
print("ID del evento:", evento_original.get("id"))
print("Tipo del evento:", evento_original.get("type", {}).get("name"))
print("Equipo del evento:", evento_original.get("team", {}).get("name"))

ID del evento: 0584ee21-e3dd-4d9f-95a0-5b5e84be25c3
Tipo del evento: Starting XI
Equipo del evento: Argentina


## 2. Construcción de la tabla y clasificación de los atributos

Un remate se identifica por el tipo de evento `Shot`. Excluimos los remates de tandas de penales, que están registrados en el período 5. Conservamos los penales cobrados durante el tiempo normal, pues no pertenecen a una tanda. A continuación construimos una fila por remate y separamos la coordenada inicial en `x` e `y`.

La próxima celda extrae los atributos requeridos desde los objetos JSON. El `for` permite ver paso a paso cómo cada evento se convierte en una fila.

In [6]:
filas_remates = []
for evento in registros_eventos:
    tipo_evento = evento.get("type", {}).get("name")
    periodo = evento.get("period")
    if tipo_evento == "Shot" and periodo != 5:
        ubicacion = evento.get("location", [None, None])
        if ubicacion is None or len(ubicacion) < 2:
            x = None
            y = None
        else:
            x = ubicacion[0]
            y = ubicacion[1]
        remate = evento.get("shot", {})
        jugador = evento.get("player", {})
        equipo = evento.get("team", {})
        parte_cuerpo = remate.get("body_part", {})
        resultado = remate.get("outcome", {})
        filas_remates.append({
            "evento_id": evento.get("id"),
            "jugador": jugador.get("name"),
            "equipo": equipo.get("name"),
            "periodo": periodo,
            "minuto": evento.get("minute"),
            "x": x,
            "y": y,
            "parte_cuerpo": parte_cuerpo.get("name"),
            "resultado": resultado.get("name")
        })

remates = pd.DataFrame(filas_remates)
print("Remates fuera de tandas de penales:", len(remates))
display(remates.head(8))

Remates fuera de tandas de penales: 30


,evento_id,jugador,equipo,periodo,minuto,x,y,parte_cuerpo,resultado
0,545c2c84-018f-4570-a01c-753823feaeac,Alexis Mac Allister,Argentina,1,4,92.4,30.0,Right Foot,Saved
1,4ad26294-8aaf-4d69-83dd-bbf9ef797b32,Rodrigo Javier De Paul,Argentina,1,7,99.2,47.9,Right Foot,Blocked
2,6d498191-05f0-432c-8764-03aea4ef9fb8,Ángel Fabián Di María Hernández,Argentina,1,16,103.1,34.6,Right Foot,Off T
3,6d527ebc-a948-4cd8-ac82-daced35bb715,Lionel Andrés Messi Cuccittini,Argentina,1,22,108.0,40.0,Left Foot,Goal
4,f227a92e-d86c-4f3d-aa75-5ccb71adcae6,Alexis Mac Allister,Argentina,1,31,94.3,23.5,Right Foot,Wayward
5,ef86f4d9-7acd-4ed0-a5ec-9129079e8fbe,Ángel Fabián Di María Hernández,Argentina,1,35,111.8,32.1,Left Foot,Goal
6,13a4889d-ad08-41be-9125-24c4c7d7879a,Rodrigo Javier De Paul,Argentina,2,48,103.3,52.3,Right Foot,Saved
7,43691970-1dad-4e11-bd29-48f640c55ea1,Julián Álvarez,Argentina,2,58,110.5,24.9,Left Foot,Saved


La tabla contiene 30 remates fuera de la tanda de penales. Cada fila representa un remate; la muestra permite comprobar que los campos anidados se tradujeron a columnas simples y que la ubicación se separó en dos coordenadas.

Consultamos el tipo de almacenamiento que eligió pandas para cada columna. Estos tipos describen cómo guarda los valores; no determinan por sí solos si el atributo representa una cantidad o una categoría.

In [7]:
tipos_almacenamiento = remates.dtypes.rename("tipo_almacenamiento").to_frame()
display(tipos_almacenamiento)

,tipo_almacenamiento
evento_id,str
jugador,str
equipo,str
periodo,int64
minuto,int64
x,float64
y,float64
parte_cuerpo,str
resultado,str


### Diccionario de datos

| Columna | Significado | Ejemplo observado | Almacenamiento usual | Tipo de atributo y escala | Unidades o categorías | Operación con sentido |
|---|---|---|---|---|---|---|
| `evento_id` | Identificador único del evento | UUID mostrado en la tabla | `object` (texto) | Categórico nominal (identificador) | Código StatsBomb | Comprobar si dos registros son el mismo evento |
| `jugador` | Nombre de quien remata | Nombre visible en la muestra | `object` | Categórico nominal | Nombres de jugadores | Contar remates por jugador |
| `equipo` | Equipo que realizó el remate | Argentina o France | `object` | Categórico nominal | Equipos del encuentro | Agrupar y contar por equipo |
| `periodo` | Parte del encuentro donde ocurrió | 1, 2, 3 o 4 | `int64` | Categórico ordinal | 1: primer tiempo; 2: segundo; 3–4: tiempos extra | Ordenar por secuencia de juego |
| `minuto` | Minuto entero reportado por StatsBomb | 12, 45, 90, etc. | `int64` | Numérico discreto, escala de intervalo | Minutos de partido; puede incluir tiempo añadido | Comparar momentos o agrupar en tramos |
| `x` | Coordenada horizontal inicial | Valor de 0 a 120 | `float64` | Numérico continuo, escala de razón | Unidades de coordenadas StatsBomb | Calcular distancia entre posiciones |
| `y` | Coordenada vertical inicial | Valor de 0 a 80 | `float64` | Numérico continuo, escala de razón | Unidades de coordenadas StatsBomb | Calcular distancia entre posiciones |
| `parte_cuerpo` | Parte del cuerpo usada para rematar | Foot, Head, Other | `object` | Categórico nominal | Categorías registradas por StatsBomb | Contar por categoría |
| `resultado` | Desenlace registrado para el remate | Goal, Saved, Off T, etc. | `object` | Categórico nominal | Categorías de resultado StatsBomb | Contar goles y otros resultados |

El tipo de almacenamiento puede variar si una columna contiene valores faltantes; por eso se consulta en la ejecución.

`periodo` se almacena como número entero, pero sus valores identifican partes del juego en un orden. No son cantidades de períodos acumulables. Por ejemplo, sumar 1 + 2 para concluir que los dos períodos equivalen al período 3 no tiene sentido: el resultado de una suma no describe duración ni intensidad. El período sirve para ordenar o agrupar fases del partido, no para operaciones aritméticas de cantidad.

`minuto` sí expresa una posición temporal aproximada y permite comparar diferencias, pero la escala no es una medida física exacta del tiempo transcurrido: el reloj de los eventos puede incluir tiempo añadido y reiniciarse entre períodos. No corresponde decir que el minuto 80 sea el doble de tarde que el minuto 40. En las coordenadas, en cambio, las diferencias entre puntos se pueden usar para calcular distancias dentro del sistema de referencia de StatsBomb.

## 3. Consultas, resúmenes y tareas de minería

La primera consulta recupera los registros de un jugador seleccionado. Luego resumimos los remates mediante conteos por equipo y por parte del cuerpo. Si el nombre no aparece en el partido, la tabla del jugador quedará vacía.

Elegimos un jugador existente en la tabla para que el ejemplo funcione con los datos seleccionados y recuperamos sus filas completas.

In [8]:
jugador_seleccionado = remates["jugador"].dropna().iloc[0]
remates_jugador = remates[remates["jugador"] == jugador_seleccionado]
print("Jugador seleccionado:", jugador_seleccionado)
print("Remates recuperados:", len(remates_jugador))
display(remates_jugador)

Jugador seleccionado: Alexis Mac Allister
Remates recuperados: 3


,evento_id,jugador,equipo,periodo,minuto,x,y,parte_cuerpo,resultado
0,545c2c84-018f-4570-a01c-753823feaeac,Alexis Mac Allister,Argentina,1,4,92.4,30.0,Right Foot,Saved
4,f227a92e-d86c-4f3d-aa75-5ccb71adcae6,Alexis Mac Allister,Argentina,1,31,94.3,23.5,Right Foot,Wayward
18,d8c05bb0-ddb1-4ff2-9dbf-ba17b00a5359,Alexis Mac Allister,Argentina,3,92,98.2,23.8,Right Foot,Off T


En este partido, Alexis Mac Allister aparece con 3 remates. La consulta recupera registros individuales y permite revisar cuándo y desde dónde remató; no resume todavía a todos los jugadores ni estima qué ocurrirá en un remate futuro.

Las siguientes tablas agrupan filas y cuentan observaciones. Son resúmenes descriptivos del partido seleccionado.

In [9]:
remates_por_equipo = remates.groupby("equipo").size().rename("cantidad_remates").to_frame()
remates_por_cuerpo = remates.groupby("parte_cuerpo").size().rename("cantidad_remates").to_frame()
print("Remates por equipo:")
display(remates_por_equipo)
print("Remates por parte del cuerpo:")
display(remates_por_cuerpo)

Remates por equipo:


,cantidad_remates
equipo,
Argentina,20
France,10


Remates por parte del cuerpo:


,cantidad_remates
parte_cuerpo,
Head,2
Left Foot,7
Right Foot,21


Argentina registró 20 remates y Francia 10. Hubo 2 remates de cabeza y 28 con el pie (7 de izquierda y 21 de derecha). Los conteos describen la frecuencia de las categorías dentro de este encuentro; no son tasas de acierto ni comparaciones de rendimiento ajustadas por posesión, minutos jugados u oportunidades. **Recuperar registros** responde quién hizo qué remates; **resumir** reduce muchas filas a conteos o medidas; **construir un modelo predictivo** requeriría aprender una regla desde atributos de entrada y comprobarla en casos que no se usaron para aprender.

### Preguntas posibles según la tarea de minería

Las preguntas siguientes se formulan para orientar un análisis futuro. No se implementan los algoritmos correspondientes.

| Tarea | Pregunta de fútbol | Observaciones | Atributos posibles | Resultado esperado | ¿Alcanza este partido? |
|---|---|---|---|---|---|
| Clasificación | ¿Un remate terminará en gol? | Un remate | Coordenadas, período, minuto, parte del cuerpo, jugador y equipo | Categoría gol / no gol | Sirve para ilustrar; hacen falta muchos partidos para entrenar y evaluar con más casos |
| Regresión | ¿Cuántos goles marcará un equipo en su próximo partido? | Un equipo-partido futuro | Remates previos, goles previos, rival, localía, descanso | Cantidad numérica de goles | No; hacen falta históricos de muchos equipos y partidos con orden temporal |
| Agrupamiento | ¿Qué perfiles de posición de remate aparecen? | Un remate | Coordenadas, distancia al arco, ángulo, parte del cuerpo | Grupo de remates con posiciones parecidas | Permite una exploración pequeña, pero varios partidos darían grupos más estables |
| Asociaciones | ¿Qué categorías de acción suelen aparecer juntas antes de un gol? | Una secuencia o posesión | Tipos de evento, pases, ubicaciones, resultado del remate | Reglas de coocurrencia entre acciones | No basta con resumir este partido; necesitamos secuencias de muchas posesiones y encuentros |
| Detección de anomalías | ¿Qué remates tienen ubicaciones inusuales respecto del conjunto? | Un remate | Coordenadas y, según la pregunta, parte del cuerpo o distancia | Casos alejados de los patrones comunes | Se puede señalar rarezas dentro del partido, pero definir normalidad exige más encuentros y contexto |

El partido sirve para practicar cómo se formulan estas preguntas y qué representa cada fila. No ofrece por sí solo volumen ni diversidad suficiente para generalizar a otras competiciones.

## 4. Formulación de un problema de predicción

**Pregunta:** ¿un remate terminará en gol? Situamos la predicción justo cuando se realiza el remate, antes de observar su desenlace. La unidad de observación es un remate individual. La variable objetivo es `gol`: vale 1 si el resultado fue `Goal` y 0 en otro caso. Como entradas posibles podemos usar `x` e `y` (posición inicial), `parte_cuerpo`, `periodo` y `equipo`; estos datos describen la situación en el momento del remate y están disponibles antes de conocer si entró el balón.

Usamos el resultado original únicamente para construir la etiqueta que queremos predecir. La tabla siguiente permite comprobar la transformación.

Creamos la columna binaria a partir de `resultado` y mostramos ambos valores lado a lado para verificar que `Goal` se convirtió en 1 y los demás desenlaces en 0.

In [10]:
remates["gol"] = 0
remates.loc[remates["resultado"] == "Goal", "gol"] = 1
comprobacion_etiqueta = remates[["resultado", "gol"]].head(12)
display(comprobacion_etiqueta)
print("Tipo de almacenamiento de gol:", remates["gol"].dtype)
print("Valores presentes:", sorted(remates["gol"].unique()))

,resultado,gol
0,Saved,0
1,Blocked,0
2,Off T,0
3,Goal,1
4,Wayward,0
5,Goal,1
6,Saved,0
7,Saved,0
8,Off T,0
9,Off T,0


Tipo de almacenamiento de gol: int64
Valores presentes: [np.int64(0), np.int64(1)]


Aunque usemos números 0 y 1, estos valores son etiquetas de dos categorías: no gol y gol. No tiene sentido calcular que un gol vale el doble que un no gol. La tarea sigue siendo de **clasificación**, porque la respuesta que buscamos es una clase discreta; no estamos estimando una cantidad numérica continua.

El campo `resultado` produciría fuga de información si se usara como entrada: describe lo ocurrido después de ejecutar el remate y es la fuente directa de la etiqueta. Un sistema que lo conociera ya sabría la respuesta. Por eso se usa solo para construir `gol` y se excluye de las variables de entrada.

Si posteriormente construyéramos un modelo, separaríamos los partidos completos en entrenamiento y evaluación. Primero necesitaríamos datos de muchos partidos y fechas, con una cantidad suficiente de goles y no goles. Aprenderíamos con encuentros anteriores y evaluaríamos en partidos posteriores que el modelo no vio. Separar por partido evita que remates del mismo encuentro aparezcan en ambos grupos y compartan contexto. Evaluar sobre los mismos casos usados para aprender solo muestra qué tan bien se ajustó a esos ejemplos; no demuestra que funcionará con partidos nuevos.

## 5. Evaluación de una estrategia sencilla

Como referencia inicial, predecimos “no gol” para todos los remates. Comparamos cada predicción con la etiqueta observada y calculamos aciertos, errores y porcentaje de aciertos. También contamos cuántos goles y no goles hubo.

Creamos una columna de predicción constante, comparamos sus valores con `gol` y contamos cada resultado. El porcentaje de aciertos es la proporción de remates clasificados correctamente en este partido.

In [11]:
remates["prediccion"] = 0
remates["acierto"] = remates["prediccion"] == remates["gol"]

cantidad_remates = len(remates)
aciertos = int(remates["acierto"].sum())
errores = cantidad_remates - aciertos
cantidad_goles = int((remates["gol"] == 1).sum())
cantidad_no_goles = int((remates["gol"] == 0).sum())
porcentaje_aciertos = 100 * aciertos / cantidad_remates

resumen_evaluacion = pd.DataFrame([
    {"medida": "Remates totales", "cantidad": cantidad_remates},
    {"medida": "Goles observados", "cantidad": cantidad_goles},
    {"medida": "No goles observados", "cantidad": cantidad_no_goles},
    {"medida": "Aciertos de la estrategia", "cantidad": aciertos},
    {"medida": "Errores de la estrategia", "cantidad": errores},
    {"medida": "Porcentaje de aciertos", "cantidad": round(porcentaje_aciertos, 2)}
])
display(resumen_evaluacion)
display(remates[["evento_id", "gol", "prediccion", "acierto"]].head(10))

,medida,cantidad
0,Remates totales,30.0
1,Goles observados,6.0
2,No goles observados,24.0
3,Aciertos de la estrategia,24.0
4,Errores de la estrategia,6.0
5,Porcentaje de aciertos,80.0


,evento_id,gol,prediccion,acierto
0,545c2c84-018f-4570-a01c-753823feaeac,0,0,True
1,4ad26294-8aaf-4d69-83dd-bbf9ef797b32,0,0,True
2,6d498191-05f0-432c-8764-03aea4ef9fb8,0,0,True
3,6d527ebc-a948-4cd8-ac82-daced35bb715,1,0,False
4,f227a92e-d86c-4f3d-aa75-5ccb71adcae6,0,0,True
5,ef86f4d9-7acd-4ed0-a5ec-9129079e8fbe,1,0,False
6,13a4889d-ad08-41be-9125-24c4c7d7879a,0,0,True
7,43691970-1dad-4e11-bd29-48f640c55ea1,0,0,True
8,9615ffbc-b513-45e6-9ccc-015234483f95,0,0,True
9,9f6285ff-9b32-4a46-a4a0-74c1ca89276c,0,0,True


En este partido hubo 24 remates que no terminaron en gol y 6 goles. La regla “siempre no gol” acertó 24 de 30 remates (80 %), principalmente porque los no goles fueron más frecuentes. Ese porcentaje no demuestra que reconozca oportunidades de gol: no identifica ningún gol y falla en todos los remates que sí terminan en gol. Por eso hay que interpretar la métrica junto con la distribución de clases y la pregunta práctica que queremos responder. El resultado futuro es incierto; esta regla no estima esa incertidumbre y no entrega una probabilidad de gol.

Estos cálculos describen un solo partido. El número y tipo de remates, la calidad de los equipos y el contexto cambian de un encuentro a otro. Para saber si la estrategia se comporta de forma parecida en general, habría que repetir la evaluación en partidos independientes de los usados para definir la regla y comparar varias medidas, incluyendo cuántos goles detecta.

## Referencias

1. [StatsBomb Open Data: competiciones, partidos, eventos y condiciones de uso](https://github.com/hudl/open-data). La documentación del repositorio solicita indicar StatsBomb como fuente y utilizar su logotipo al compartir análisis.
2. [Especificación de StatsBomb Open Data](https://github.com/statsbomb/open-data/blob/master/doc/StatsBomb%20Open%20Data%20Specification%20v1.1.pdf): estructura y significado de los registros de eventos.
3. [Documentación de pandas](https://pandas.pydata.org/docs/): lectura, selección y resumen de tablas.
4. [Archivo de competiciones](https://raw.githubusercontent.com/hudl/open-data/4b73468fc5b0f1950f9f66fada70ad3a4f9327cb/data/competitions.json) y [archivo de partidos del Mundial 2022](https://raw.githubusercontent.com/hudl/open-data/4b73468fc5b0f1950f9f66fada70ad3a4f9327cb/data/matches/43/106.json), consultados desde una versión fija del repositorio.

**Atribución:** Datos proporcionados por StatsBomb a través de StatsBomb Open Data. El logotipo pertenece a StatsBomb. Este análisis es un trabajo educativo independiente y no está respaldado por StatsBomb.